In [1]:
import numpy as np

In [2]:
# extract audio features using librosa
def extract_features(audio_file):
  import librosa

  y, sr = librosa.load(audio_file, sr=None)
  mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
  chroma = librosa.feature.chroma_stft(y=y, sr=sr)
  spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
  rms = librosa.feature.rms(y=y)[0]

  features = {
      'mfcc': np.mean(mfcc, axis=1),
      'chroma': np.mean(chroma, axis=1),
      'spectral_centroid': np.mean(spectral_centroid),
      'rms': np.mean(rms),
  }

  return features

In [ ]:
!unzip train.zip

In [3]:
class_labels = ['guitar', 'harp', 'piano', 'strings']
num_classes = len(class_labels)

In [4]:
# build dataset from audio files under train for each instrument

input_x = []
output_y = []

import os

for instrument in class_labels:
  dir_path = os.path.join("train", instrument)
  for filename in os.listdir(dir_path):
    audio_file = os.path.join(dir_path, filename)
    if os.path.isfile(audio_file):

      # extract audio features
      features = extract_features(audio_file)
      feature_vector = np.concatenate([features['mfcc'], features['chroma'], [features['spectral_centroid']], [features['rms']]])
      input_x.append(feature_vector)
      output_y.append(instrument)

input_x = np.array(input_x)
output_y = np.array(output_y)

In [5]:
input_x.shape

(249, 27)

In [6]:
output_y.shape

(249,)

In [7]:
# split data for training and testing

from sklearn.model_selection import train_test_split
train_x, test_x, train_y, test_y = train_test_split(input_x, output_y, shuffle=True, test_size=0.2)

In [8]:
# choose and run ML model

from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier().fit(train_x, train_y)
model.score(test_x, test_y)

0.98

In [9]:
# define neural network architecture

import keras
from keras import layers

inputs = keras.Input(shape=(27, ))
dense1 = layers.Dense(32, activation="linear")(inputs)
dense2 = layers.Dense(32, activation="linear")(dense1)

outputs = layers.Dense(num_classes, activation="softmax")(dense2)
model = keras.Model(inputs=inputs, outputs=outputs)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 27)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,084 (8.14 KB)

 Trainable params: 2,084 (8.14 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

In [11]:
# convert class labels from string to number

from sklearn.preprocessing import LabelEncoder
z = LabelEncoder().fit_transform(output_y)

In [12]:
model.fit(input_x, z, validation_split=0.2, epochs=20, shuffle=True)

Epoch 1/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - accuracy: 0.1612 - loss: 305.8607 - val_accuracy: 0.0200 - val_loss: 33.1105
Epoch 2/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.3319 - loss: 89.4799 - val_accuracy: 0.0000e+00 - val_loss: 110.8035
Epoch 3/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6456 - loss: 41.2698 - val_accuracy: 0.4400 - val_loss: 13.5790
Epoch 4/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7696 - loss: 17.1086 - val_accuracy: 0.1600 - val_loss: 19.7627
Epoch 5/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6505 - loss: 17.1111 - val_accuracy: 0.3600 - val_loss: 15.7898
Epoch 6/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7798 - loss: 11.4906 - val_accuracy: 0.9400 - val_loss: 1.1344
Epoch 7/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8757 - loss: 8.4205 - val_accuracy: 0.8600 - val_loss: 3.2017
Epoch 8/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8052 - loss: 9.2268 - val_accuracy: 0.6600 - 